# Carregando dados e imports

In [1]:
import shutil
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

RAW_CSV = Path("../data/raw/covid.csv")
if not RAW_CSV.exists():
    dataset_dir = Path(kagglehub.dataset_download("meirnizri/covid19-dataset"))
    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(next(dataset_dir.glob("*.csv")), RAW_CSV)
raw = pd.read_csv(RAW_CSV)

print(f"Quantidade de registros: {len(raw):,}")
raw.head()

Quantidade de registros: 1,048,575


,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,DATE_DIED,INTUBED,PNEUMONIA,AGE,PREGNANT,DIABETES,...,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL,ICU
0,2,1,1,1,03/05/2020,97,1,65,2,2,...,2,2,1,2,2,2,2,2,3,97
1,2,1,2,1,03/06/2020,97,1,72,97,2,...,2,2,1,2,2,1,1,2,5,97
2,2,1,2,2,09/06/2020,1,2,55,97,1,...,2,2,2,2,2,2,2,2,3,2
3,2,1,1,1,12/06/2020,97,2,53,2,2,...,2,2,2,2,2,2,2,2,7,97
4,2,1,2,1,21/06/2020,97,2,68,97,1,...,2,2,1,2,2,2,2,2,3,97


# Tratando as variáveis

As 17 variáveis da tabela de `docs/questionnaire.md`: binárias em 0 = Não / 1 = Sim (no bruto: 1 = Sim, 2 = Não; códigos de ausência 97/98/99 viram 0), teste COVID com a escala invertida (quanto maior, mais indica COVID diagnosticado) e ÓBITO derivado de `DATE_DIED` (`9999-99-99` = não faleceu).

In [ ]:
RENAME = {
    "AGE": "IDADE",
    "USMER": "NÍVEL_ATENDIMENTO",
    "INTUBED": "INTUBADO",
    "PNEUMONIA": "PNEUMONIA",
    "PREGNANT": "GRÁVIDA",
    "DIABETES": "DIABETES",
    "COPD": "DPOC",
    "ASTHMA": "ASMA",
    "INMSUPR": "IMUNOSSUPRESSÃO",
    "HIPERTENSION": "HIPERTENSÃO",
    "OTHER_DISEASE": "OUTRAS_DOENÇAS",
    "CARDIOVASCULAR": "DOENÇA_CARDIOVASCULAR",
    "OBESITY": "OBESIDADE",
    "RENAL_CHRONIC": "DOENÇA_RENAL_CRÔNICA",
    "TOBACCO": "TABAGISMO",
    "ICU": "UTI",
    "CLASIFFICATION_FINAL": "CLASSIFICAÇÃO_FINAL_TESTE_COVID",
}
BINARIAS = [c for c in RENAME.values() if c not in ("IDADE", "CLASSIFICAÇÃO_FINAL_TESTE_COVID")]

df = raw.rename(columns=RENAME)[list(RENAME.values())].copy()

for col in BINARIAS:
    df[col] = df[col].map({1: 1, 2: 0}).fillna(0)  # 97/98/99 (ausente) -> 0

df.loc[df["IDADE"].isin([97, 98, 99]), "IDADE"] = np.nan
df["IDADE"] = df["IDADE"].fillna(df["IDADE"].median())

df["CLASSIFICAÇÃO_FINAL_TESTE_COVID"] = 8 - df["CLASSIFICAÇÃO_FINAL_TESTE_COVID"]

df["ÓBITO"] = (raw["DATE_DIED"] != "9999-99-99").astype(int)

print(f"{df.shape[1] - 1} preditoras | taxa de óbito: {df['ÓBITO'].mean():.2%}")
df.head()

17 preditoras | taxa de óbito: 7.34%


,IDADE,NÍVEL_ATENDIMENTO,INTUBADO,PNEUMONIA,GRÁVIDA,DIABETES,DPOC,ASMA,IMUNOSSUPRESSÃO,HIPERTENSÃO,OUTRAS_DOENÇAS,DOENÇA_CARDIOVASCULAR,OBESIDADE,DOENÇA_RENAL_CRÔNICA,TABAGISMO,UTI,CLASSIFICAÇÃO_FINAL_TESTE_COVID,ÓBITO
0,65.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1
1,72.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,3,1
2,55.0,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1
3,53.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
4,68.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1


# Salvando dataset processado

In [3]:
saida = Path("../data/processed/covid.csv")
saida.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(saida, index=False)